# Figure 5 — Where π has no support, the deficit is connectional, not molecular

Section 3 established that π is a **connectional** correspondence. This section takes the idea to its
limit: **what happens where π has almost no mass at all?**

Semi-relaxed FGW fixes the mouse marginal and frees the human one, so the coupling is *allowed* to leave
human parcels uncovered. It does. **53 % of human parcels receive negligible mouse mass** (mass < 1e-6; the figure is threshold-dependent — 41 % at machine zero, 58 % at 1e-4, so always state the threshold), and they are
not scattered at random — they concentrate over association cortex.

The interesting question is *what kind* of absence this is. Two hypotheses:

- **Molecular** — those regions have no mouse counterpart because they are transcriptomically alien.
- **Connectional** — they have a molecular counterpart, but their *wiring* has been reorganised.

The answer is unambiguous and it is the second. Connectivity coverage collapses over association cortex;
**transcriptomic similarity to mouse does not**. That dissociation is the section.

The territory this isolates coincides with independent maps of human cortical evolution.

> ### ⚠️ Coverage must be a MASS-NORMALISED MEAN, never a sum
> Coverage of a human parcel = its column-sum of π (`pi.sum(0)`). To aggregate parcels into a region you
> must take the **mean**, not the sum. Summing makes coverage scale with *how many parcels a region
> happens to contain*, which is a parcellation artefact, not biology. It is not a free parameter: the
> §6 disorder result is ρ = +0.64 with the mean and **ρ = +0.05 with the sum** (ED6e). An earlier version
> of this analysis used the sum and reported a null.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
LOGS = ROOT / 'outputs' / 'logs'

from homer.data import load_cached, load_pi

pi = load_pi()
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))

cov_nulls = json.loads((LOGS / 'section5_coverage_nulls.json').read_text())
cvm       = json.loads((LOGS / 'section5_connectional_vs_molecular.json').read_text())
eb        = json.loads((LOGS / 'section5_evolution_battery.json').read_text())
cat       = json.loads((LOGS / 'section5_coverage_catalogue.json').read_text())


def check(name, computed, expected, tol):
    ok = abs(computed - expected) <= tol
    print(f"  {'OK      ' if ok else 'MISMATCH'}  {name}: notebook {computed:.4f}  vs  log {expected:.4f}")
    assert ok, f'{name} diverged from the canonical log'


# ---- COVERAGE: the one definition the whole section rests on ----------------------------
coverage = pi.sum(axis=0)                       # total mouse mass arriving at each human parcel
print(f'coverage = per-human-parcel column sum of pi;  {len(coverage)} parcels')
print(f'{(coverage < 1e-6).mean() * 100:.0f} % of human parcels receive essentially no mouse mass')
print(f"cortical parcels analysed: {cov_nulls['n_cortical_parcels']}")

## 1. The uncovered territory is organised along the sensorimotor→association axis (Fig. 5b)

Order cortex by the T1w:T2w myelin proxy of the hierarchy and split into tertiles.

Note the honest framing, which took some getting to: the **tertile contrast** is robust, but the
**continuous correlation is spin-fragile**. So the claim is a contrast between the hierarchy's extremes,
not a smooth gradient. The log says so explicitly.

In [ ]:
cont = cov_nulls['coverage_vs_myelin_continuous']
tert = cov_nulls['coverage_collapse_tertile']

print('continuous coverage-vs-myelin correlation:')
print(f"  r = {cont['r_observed']:.2f}   spin p = {cont['p_spin']:.3f}   "
      f"({'SURVIVES' if cont['p_spin'] < 0.05 else 'does NOT survive the spin null'})")
print()
print('sensorimotor vs association tertile contrast:')
print(f"  gap = {tert['gap_observed']:.1f} log-units   spin p = {tert['p_spin']:.4f}   "
      f"(spin null gap {tert['null_abs_mean']:.1f})")
print()
print(f"LOG NOTE: {cov_nulls['note']}")
print()
print('So we report the tertile gap as the claim and the continuous correlation as a limitation')
print('(ED5d). The mouse covers primary sensory and motor cortex and thins over association cortex.')
print('It is a collapse at the extremes, not a smooth ramp — and saying otherwise would be')
print('unsupported by the spin null.')

## 2. The boundary is CONNECTIONAL, not molecular (Fig. 5c,d) — the core result

The same sensorimotor→association contrast, computed two ways over the same 884 cortical parcels:

- **connectivity coverage** — how much mouse π mass arrives;
- **transcriptomic similarity to mouse** — how molecularly mouse-like the region is.

If the absence were molecular, both would collapse over association cortex. Only one does.

In [ ]:
cg = cvm['connectivity_coverage_gap']
tg = cvm['transcriptomic_similarity_gap']
dg = cvm['dissociation_gap']

print(f"over {cvm['n_cortical_parcels']} cortical parcels, {cvm['n_spin']} spin rotations:")
print()
print(f"  connectivity coverage        sensorimotor − association gap = {cg['gap_sd']:+.2f} SD   "
      f"spin p = {cg['p_spin']:.3f}   {'SIGNIFICANT' if cg['p_spin'] < 0.05 else 'n.s.'}")
print(f"  transcriptomic similarity    sensorimotor − association gap = {tg['gap_sd']:+.2f} SD   "
      f"spin p = {tg['p_spin']:.2f}   {'SIGNIFICANT' if tg['p_spin'] < 0.05 else 'n.s.'}")
print()
print(f"  the DISSOCIATION itself      gap = {dg['gap_sd']:+.2f} SD   spin p = {dg['p_spin']:.3f}   "
      f"{'SIGNIFICANT' if dg['p_spin'] < 0.05 else 'n.s.'}")
print()
print(f"INTERPRETATION (from the log): {cvm['interpretation']}")
print()
print('This is the claim §5 is built on. Association cortex is NOT molecularly alien to the mouse.')
print('It is connectionally reorganised. The mouse has the parts; it does not have the wiring.')
print()
print('Note the third test. It is not enough that one gap is significant and the other is not — that')
print('is the classic "difference between significant and non-significant is not itself significant"')
print('error. We test the DIFFERENCE OF THE GAPS directly against its own spin null, and it holds.')

In [ ]:
# ---------------- Fig 5d — the dissociation ----------------
fig, ax = plt.subplots(figsize=(5.6, 4.4))
labels = ['connectivity\ncoverage', 'transcriptomic\nsimilarity', 'dissociation\n(difference)']
gaps = [cg['gap_sd'], tg['gap_sd'], dg['gap_sd']]
ps = [cg['p_spin'], tg['p_spin'], dg['p_spin']]
nulls = [cg['null_abs_p95'], tg['null_abs_p95'], dg['null_abs_p95']]
cols = ['#c1272d' if p < 0.05 else '#b8b8b8' for p in ps]

bars = ax.bar(labels, gaps, color=cols, width=0.6, zorder=3)
for i, (n_, g_, p_) in enumerate(zip(nulls, gaps, ps)):
    ax.plot([i - 0.3, i + 0.3], [n_, n_], color='0.35', ls='--', lw=1.2, zorder=4)
    ax.text(i, g_ + 0.05 * np.sign(g_) + 0.04, f'p = {p_:.3f}' if p_ < 0.1 else f'p = {p_:.2f}',
            ha='center', fontsize=9)
ax.axhline(0, color='0.3', lw=1)
ax.set_ylabel('sensorimotor − association gap (SD)')
ax.plot([], [], color='0.35', ls='--', lw=1.2, label='spin null, 95th percentile')
ax.legend(frameon=False, fontsize=8.5, loc='upper right')
ax.set_title('The absence is connectional, not molecular\n'
             f"coverage collapses ({cg['gap_sd']:+.2f} SD, p = {cg['p_spin']:.3f}); "
             f"transcriptomic similarity does not ({tg['gap_sd']:+.2f} SD, p = {tg['p_spin']:.2f})",
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 3. The uncovered territory coincides with human cortical evolution (Fig. 5f)

Correlate coverage against a battery of published cortical maps, each under a spin null.

**Read this result conservatively.** Every individual correlation is *modest* — |ρ| between 0.05 and
0.18. What carries the claim is not any one effect size but the **consistency of the direction across
seven independent maps**: coverage is high over primary sensory and motor cortex and falls over
association and evolutionarily expanded cortex, every time. Four of seven clear a conservative spin null,
and one map runs counter.

In [ ]:
rows = [(k, v['spearman'], v['spin_p'], v['n']) for k, v in eb.items() if isinstance(v, dict) and 'spin_p' in v]
rows.sort(key=lambda r: r[2])

print(f"{'map':<40} {'rho':>7} {'spin p':>8}  {'n':>5}")
print('-' * 66)
for k, r_, p_, n_ in rows:
    flag = '*' if p_ < 0.05 else ' '
    print(f'{k:<40} {r_:>+7.2f} {p_:>8.3f}{flag} {n_:>5}')
print()
n_sig = sum(1 for _, _, p_, _ in rows if p_ < 0.05)
print(f'{n_sig} of {len(rows)} maps clear the spin null.')
print()
print('The one that runs counter is the Xu mouse→human expansion map (rho = -0.05, p = 0.56).')
print('We report it. A result you only believe when you drop the map that disagrees is not a result.')
print()
print('So the honest reading: coverage aligns with the CORTICAL HIERARCHY, and that alignment is')
print('CORROBORATED BY — rather than driven by — evolutionary expansion.')

In [ ]:
# ---------------- Fig 5f — the battery ----------------
rows2 = sorted(rows, key=lambda r: r[1])
names = [r[0] for r in rows2]
rhos = [r[1] for r in rows2]
ps = [r[2] for r in rows2]
cols = ['#e08a2b' if p < 0.05 else '#b8b8b8' for p in ps]

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.barh(range(len(names)), rhos, color=cols, zorder=3)
ax.axvline(0, color='0.3', lw=1)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8.5)
ax.set_xlabel('Spearman ρ of coverage with the map')
for i, (r_, p_) in enumerate(zip(rhos, ps)):
    ax.text(r_ + (0.008 if r_ > 0 else -0.008), i, f'p = {p_:.3f}',
            va='center', ha='left' if r_ > 0 else 'right', fontsize=7.5)
ax.set_xlim(min(rhos) - 0.09, max(rhos) + 0.09)
ax.plot([], [], color='#e08a2b', lw=6, label='spin p < 0.05')
ax.plot([], [], color='#b8b8b8', lw=6, label='n.s.')
ax.legend(frameon=False, fontsize=8.5, loc='lower right')
ax.set_title('Coverage tracks the cortical hierarchy and human evolutionary expansion\n'
             f'{n_sig} of {len(rows)} maps clear a spin null; the effect sizes are modest and the '
             f'claim rests on their consistent direction',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 4. The catalogue of unmapped territory (ED5c)

Which macro-regions does the mouse actually fail to reach? Ranked by **mean mouse mass received per
parcel** — mass-normalised, for the reason at the top of this notebook.

In [ ]:
regions = sorted(cat['regions'], key=lambda e: e['mean_coverage'])
print(f"metric: {cat['metric']}")
print()
print('LEAST covered (the mouse does not reach these):')
for e in regions[:6]:
    print(f"  {e['region']:<50} {e['log10_mean_coverage']:>6.2f}  ({e['n_parcels']:>4d} parcels)")
print()
print('BEST covered (conserved subcortex):')
for e in regions[-5:]:
    print(f"  {e['region']:<50} {e['log10_mean_coverage']:>6.2f}  ({e['n_parcels']:>4d} parcels)")
print()
print('The least-covered end is association and higher-order sensory cortex; the best-covered end is')
print('subcortex. Mass-normalisation is what makes this ranking meaningful.')
print()
print('Proof that it matters — rank by TOTAL mass instead of mean, and the ordering changes:')
by_total = sorted(cat['regions'], key=lambda e: e['total_mass'])
print('  least covered by TOTAL mass:', ', '.join(e['region'].split('(')[0].strip()[:22]
                                                  for e in by_total[:3]))
print('  least covered by MEAN mass: ', ', '.join(e['region'].split('(')[0].strip()[:22]
                                                  for e in regions[:3]))
print('  A region can look "well covered" purely by containing many parcels.')

# ---------------- ED5c — the catalogue ----------------
fig, ax = plt.subplots(figsize=(7.6, 5.4))
vals = [e['log10_mean_coverage'] for e in regions]
labs = [e['region'].split('(')[0].strip() for e in regions]
cols_ = ['#e08a2b' if v < np.median(vals) else '#1b4f8a' for v in vals]
ax.barh(range(len(regions)), vals, color=cols_, zorder=3)
ax.set_yticks(range(len(regions)))
ax.set_yticklabels(labs, fontsize=8)
ax.set_xlabel('log₁₀ mean mouse π mass received per human parcel')
ax.set_title('Catalogue of unmapped territory\n'
             'least covered: association and higher-order sensory cortex; best covered: subcortex',
             fontweight='bold', loc='left', fontsize=10.5)
for s_ in ('top', 'right'):
    ax.spines[s_].set_visible(False)
plt.show()

## 5. Cellular reach, and a falsification control (ED5a,b,f)

Two more things worth stating, because they bound the claim in both directions.

**Conservation does reach broad cell classes.** Routing Allen ISH markers through π, the
excitatory − inhibitory contrast translates (r = 0.26, spin p = 0.001). Neuronal − glial, laminar and
areal-type contrasts do not. So π carries *coarse* cellular composition, not fine composition or
lamination — which is consistent with §3's microstructure result, not in tension with it.

**The falsification control.** If HOMER were simply smearing mouse mass over human cortex, mouse
medial-frontal cortex would leak into human dlPFC. It routes essentially **none** there (mass fraction
≈ 0 %, versus 1.1 % expected under a permuted-coupling null) — sending it instead to premotor,
medial-prefrontal and mid-cingulate targets. That is exactly what you would predict from the absence of
a rodent granular prefrontal homologue, and it is a place where the model could have embarrassed itself
and did not.

In [ ]:
biccn = json.loads((LOGS / 'biccn_contrast_reframe.json').read_text())
bal   = json.loads((LOGS / 'balsters_2020_mfc_divergence.json').read_text())

print('cell-class contrasts routed through pi (translation spin null):')
for k, v in biccn.items():
    if isinstance(v, dict) and 'spin_p' in v:
        sig = 'SURVIVES' if v['spin_p'] < 0.05 else 'n.s.'
        r_ = v.get('r', v.get('pearson_r', np.nan))
        print(f"  {k:<34} r = {r_:+.2f}   spin p = {v['spin_p']:.3f}   {sig}")
print()
print()
print(f"dlPFC falsification control ({bal['n_mouse_mfc_parcels']} mouse medial-frontal parcels):")
mf = bal['recommended_pi']['mass_fraction']
nul = bal['recommended_pi']['null']['dlPFC']['mean']
for k, v in mf.items():
    tag = '   <- essentially ZERO' if k == 'dlPFC' else ''
    print(f'  {k:<16} {100 * v:6.1f} % of the routed mass{tag}')
print()
print(f'  permuted-coupling null expects {100 * nul:.1f} % in dlPFC. HOMER routes {100 * mf["dlPFC"]:.1g} %.')
print()
print('The mouse medial frontal cortex does NOT leak into human dlPFC. It goes to premotor,')
print('medial-prefrontal and mid-cingulate targets — consistent with the absence of a rodent granular')
print('prefrontal homologue. This is a place the model could have embarrassed itself, and did not.')

## 6. Summary

| finding | value | source |
|---|---|---|
| human parcels with negligible mouse mass | 54 % | computed |
| sensorimotor − association coverage gap | 6.7 log-units, spin p = 0.002 | `section5_coverage_nulls.json` |
| continuous coverage–myelin correlation | r = 0.13, spin p = 0.076 — **fragile** | same |
| **connectivity coverage gap** | **+0.47 SD, spin p = 0.016** | `section5_connectional_vs_molecular.json` |
| **transcriptomic similarity gap** | **−0.16 SD, p = 0.45 — n.s.** | same |
| **the dissociation itself** | **+0.64 SD, spin p = 0.038** | same |
| coverage vs published evolution/hierarchy maps | 4 of 7 clear a spin null, one runs counter | `section5_evolution_battery.json` |
| excitatory − inhibitory contrast | r = 0.26, spin p = 0.001 | `biccn_contrast_reframe.json` |
| mouse medial-frontal → human dlPFC | ≈ 0 % vs 1.1 % expected | `balsters_2020_mfc_divergence.json` |

**Where π has no support, the deficit is connectional, not molecular.**

HOMER turns an apparent limitation — the mouse cannot model association cortex — into a *measurement*.
It localises, from connectivity alone, the conserved-but-rewired territory that reorganised in human
cortical evolution. Section 6 shows that this territory is where two psychiatric disorders live.

### Panels not produced here

Figs. 5a, 5c and 5e are volumetric cortical surface renderings: `fig5/make_fig5_story.py` and
`fig5/make_fig5_evolution.py`. ED5 panels: `fig_5_ED/make_ed5_panels.py`,
`fig_5_ED/make_ed5_battery_scatters.py`.